In [0]:
%run "../includes/librerias"

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
# 1. leemos el archivo csv

#definimos los tipos de datos del archivo
language_schema = StructType([
    StructField("languageId", IntegerType(), True),
    StructField("languageCode", StringType(), True),
    StructField("languageName", StringType(), True)
 ] )

#cargamos el df indicando los tipos de datos y formato del archivo como la cavecera
language_df = spark.read\
    .option("header", True)\
    .schema(language_schema)\
    .csv(f"{bronze_folder_path}/{file_date}/language.csv")

language_df.printSchema()

In [0]:
# Paso 2 - Seleccionar las columnas que se requieren

language_selected_df = language_df.select("languageId", "languageName")
language_selected_df.printSchema()


In [0]:
# Paso 3 - Renombrar Columnas

language_renamed_df = language_selected_df\
    .withColumnRenamed("languageId", "language_id")\
    .withColumnRenamed("languageName", "language_name")

display(language_renamed_df)

In [0]:
# Paso 4 - Añadir columnas a una tabla

language_final_df = add_ingestion_date(language_renamed_df)
language_final_df = add_env(language_final_df)
language_final_df = add_file_date(language_final_df)

display(language_final_df)
language_final_df.printSchema()

In [0]:
# Paso 5 - Guardar datos en datalake en formato parket

language_final_df.write.mode("overwrite").format("delta").saveAsTable("movie_silver.languages")


In [0]:
%sql
select * from movie_silver.languages;

In [0]:
dbutils.notebook.exit("El notebook 02.Ingestion File Language, termino correctamente")
